In [4]:
from google.colab import files
uploaded = files.upload()

Saving StudentPerformanceFactors.csv to StudentPerformanceFactors.csv


In [5]:
import pandas as pd

df = pd.read_csv("StudentPerformanceFactors.csv")
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [6]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               1015 non-null   int64 
 1   Attendance                  1015 non-null   int64 
 2   Parental_Involvement        1015 non-null   object
 3   Access_to_Resources         1015 non-null   object
 4   Extracurricular_Activities  1015 non-null   object
 5   Sleep_Hours                 1015 non-null   int64 
 6   Previous_Scores             1015 non-null   int64 
 7   Motivation_Level            1015 non-null   object
 8   Internet_Access             1015 non-null   object
 9   Tutoring_Sessions           1015 non-null   int64 
 10  Family_Income               1015 non-null   object
 11  Teacher_Quality             1005 non-null   object
 12  School_Type                 1015 non-null   object
 13  Peer_Influence              1015 non-null   obje

,0
Hours_Studied,0
Attendance,0
Parental_Involvement,0
Access_to_Resources,0
Extracurricular_Activities,0
Sleep_Hours,0
Previous_Scores,0
Motivation_Level,0
Internet_Access,0
Tutoring_Sessions,0


In [7]:
df = df.fillna("Unknown")

In [8]:
target = "Exam_Score"

X = df.drop(columns=[target])
y = df[target]

In [9]:
X = pd.get_dummies(X, drop_first=True)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(alpha=0.1),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(eval_metric="rmse")
}

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    })

    trained_models[name] = model

    print("\n", name)
    print("R2:", r2_score(y_test, preds))


 Linear Regression
R2: 0.5469093673588347

 Lasso
R2: 0.5239319781717013

 Random Forest
R2: 0.40419163906250943

 XGBoost
R2: 0.4232168197631836


In [14]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df.sort_values(by="R2", ascending=False)

,Model,MAE,RMSE,R2
0,Linear Regression,0.783831,2.861821,0.546909
1,Lasso,0.994776,2.933489,0.523932
3,XGBoost,1.418496,3.228912,0.423217
2,Random Forest,1.569049,3.281733,0.404192


In [15]:
best_model_name = results_df.sort_values(by="R2", ascending=False).iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Linear Regression


In [16]:
import pickle

pickle.dump(best_model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(X.columns, open("features.pkl", "wb"))

print("Saved successfully")

Saved successfully
